# PPMI_DemRKL
Copied from the original project. Paths are relative to the main folder.
Original DemRKG generation and training settings are preserved.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'PPMI'
METHOD = 'DemRKL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
import pandas as pd

def batch_inspect_ontology_ppmi():
    print("==================================================")
    print("🛠️ 步骤 1: 启动 PrimeKG 本体对齐批量检修工具 (PPMI专版)...")
    kg_df = pd.read_csv('kg.csv', low_memory=False)
    print("   ✅ 成功加载 kg.csv。")

    search_df = kg_df[kg_df['x_type'].isin(['disease', 'phenotype', 'effect/phenotype'])].copy()

    # 修正：增加帕金森病四大核心运动症状的专业医学词汇
    search_targets = {
        "核心疾病 (Parkinson)": ["parkinson"],
        "认知与记忆 (Cognition/Memory)": ["cognitive", "memory", "dementia", "amnesia"],
        
        # --- 新增核心症状检索 ---
        "核心症状-震颤 (Tremor)": ["tremor"],
        "核心症状-强直 (Rigidity)": ["rigidity"],
        "核心症状-运动迟缓 (Bradykinesia)": ["bradykinesia"],
        "核心症状-异动症 (Dyskinesia)": ["dyskinesia"],
        
        # --- 保留原有的部位检索作为辅助参考 ---
        "面部表情 (Facial)": ["facial", "face"],
        "手指敲击与手部运动 (Finger/Hand)": ["finger", "hand", "motor", "movement"],
        "旋前旋后 (Pronation)": ["pronation", "supination"],
        "步态与姿势 (Gait & Posture)": ["gait", "posture", "stability", "balance"],
        "起立与下肢敏捷 (Leg/Chair)": ["leg", "limb", "muscle", "weakness"]
    }

    print("\n==================================================")
    print("🔍 步骤 3: 开始按医学词根进行批量模糊检索...\n")

    for concept, keywords in search_targets.items():
        print(f"▶ 正在检索【{concept}】...")
        print(f"   使用的词根: {keywords}")
        
        pattern = '|'.join(keywords)
        mask = search_df['x_name'].str.contains(pattern, case=False, na=False)
        results = search_df[mask]['x_name'].unique()
        
        if len(results) > 0:
            print(f"   ✅ 找到 {len(results)} 个候选节点，前 10 个最精准的候选如下：")
            # 按照字符串长度排序，优先展示最简短、最基础的节点
            sorted_results = sorted(results, key=len)
            for i, res in enumerate(sorted_results[:10]):
                print(f"      [{i+1}] {res}")
        else:
            print(f"   ❌ 使用这些词根未能找到任何节点。")
        print("")

if __name__ == "__main__":
    batch_inspect_ontology_ppmi()